# Student Lifestyle Analytics — Exploratory Data Analysis

**Project:** Predicting Academic Stress and Burnout  
**Dataset:** StressLevelDataset (psychometric survey, 1 100 students, 20 features)  
**Approach:** Hybrid K-Means Clustering + Per-Cluster Classification  

This notebook walks through:
1. Dataset loading and validation
2. Preprocessing pipeline
3. Feature engineering
4. Exploratory visualisations
5. Train/test split, imputation, scaling
6. Clustering preview
7. Key findings summary

In [ ]:
%matplotlib inline
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# Add project root to path so we can import from src/
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
print('Environment ready.')

## 1. Load Data

We load **StressLevelDataset.csv** (rxnach, Kaggle) — a psychometric survey of 1 100 students with 20 features covering anxiety, depression, sleep quality, social support, peer pressure, and more.  
If the CSV is absent the loader automatically generates a **2 000-row synthetic dataset** with the same schema.

In [ ]:
from data_loader import load_data

raw_csv = project_root / 'data' / 'raw' / 'StressLevelDataset.csv'
df_raw = load_data(str(raw_csv) if raw_csv.exists() else None, n_samples=2000)
print(f'\nShape: {df_raw.shape}')
df_raw.head()

### Class distribution check

In [ ]:
print('Stress level distribution:')
print(df_raw['stress_level'].value_counts())
print('\nBurnout distribution:')
print(df_raw['burnout'].value_counts())

## 2. Preprocessing

`preprocess_data` is **stateless** — it only applies operations that require no statistics learned from data:
- **Duplicate removal**
- **LabelEncoder** for targets (`stress_level` → 0/1/2, `burnout` → 0/1)

Mean imputation and IQR outlier capping are intentionally deferred to Step 5 (`fit_transform_cleaning`),  
where they are fitted on the training set only to prevent data leakage into the test set.

In [ ]:
from preprocessing import preprocess_data

df, artifacts = preprocess_data(df_raw)
print(f'Post-preprocessing shape: {df.shape}')
df.describe().T

## 3. Feature Engineering

Five domain-derived composite features are constructed from the original (pre-scaled) psychometric values:

| Feature | Formula summary |
|---|---|
| `psychological_burden` | `anxiety×0.35 + depression×0.35 − self_esteem×0.30` |
| `academic_stress` | `study_load×0.40 + peer_pressure×0.30 − academic_performance×0.30` |
| `wellbeing_index` | `sleep_quality + social_support + safety + basic_needs − headache − breathing_problem` |
| `environmental_stress` | `noise×0.40 + bullying×0.40 − living_conditions×0.20` |
| `support_deficit` | `peer_pressure + future_career_concerns − social_support − teacher_student_relationship` |

In [ ]:
from feature_engineering import create_features

df = create_features(df)
engineered_cols = [
    'psychological_burden', 'academic_stress', 'wellbeing_index',
    'environmental_stress', 'support_deficit',
]
df[engineered_cols].describe().T

## 4. Exploratory Visualisations

> All plots below are generated on **pre-scaled** data so axes retain interpretable units.

### 4.1 Correlation Heatmap

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(16, 12))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.4, annot_kws={'size': 6}, ax=ax)
ax.set_title('Correlation Heatmap — All Numeric Features', fontsize=14)
plt.tight_layout()
plt.show()

**Finding:** `anxiety_level` and `depression` show the strongest positive correlation with `stress_level_encoded`. `self_esteem` and `sleep_quality` are negatively correlated with stress — acting as protective factors. The engineered `psychological_burden` feature amplifies these relationships.

### 4.2 Distribution Histograms

In [ ]:
cols = ['anxiety_level', 'depression', 'self_esteem', 'sleep_quality', 'study_load', 'social_support']
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
for i, col in enumerate(cols):
    axes[i].hist(df[col].dropna(), bins=35, color='steelblue', edgecolor='white', alpha=0.85)
    axes[i].set_title(f'Distribution of {col}', fontsize=11)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')
plt.suptitle('Primary Psychometric Feature Distributions', fontsize=14)
plt.tight_layout()
plt.show()

### 4.3 Stress Level Distribution

In [ ]:
palette = {'Low': '#2ecc71', 'Medium': '#f39c12', 'High': '#e74c3c'}
order = ['Low', 'Medium', 'High']
counts = df['stress_level'].value_counts().reindex(order)

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(counts.index, counts.values, color=[palette[o] for o in order], width=0.5, edgecolor='white')
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, str(val), ha='center', fontsize=11)
ax.set_title('Stress Level Class Distribution', fontsize=13)
ax.set_xlabel('Stress Level')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

### 4.4 Burnout Distribution

In [ ]:
counts_b = df['burnout'].value_counts()
colors_b = ['#e74c3c' if k == 'Yes' else '#2ecc71' for k in counts_b.index]

plt.figure(figsize=(6, 6))
plt.pie(counts_b.values, labels=counts_b.index, autopct='%1.1f%%',
        colors=colors_b, startangle=140, wedgeprops={'edgecolor': 'white'})
plt.title('Burnout Distribution', fontsize=13)
plt.show()

### 4.5 Boxplots by Stress Level

In [ ]:
numeric_cols = ['anxiety_level', 'depression', 'self_esteem', 'sleep_quality', 'study_load', 'social_support']
stress_order = ['Low', 'Medium', 'High']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()
for i, col in enumerate(numeric_cols):
    groups = [df[df['stress_level'] == lvl][col].dropna() for lvl in stress_order]
    bp = axes[i].boxplot(groups, labels=stress_order, patch_artist=True)
    for patch, lvl in zip(bp['boxes'], stress_order):
        patch.set_facecolor(palette[lvl])
        patch.set_alpha(0.65)
    axes[i].set_title(f'{col} by Stress Level', fontsize=10)
    axes[i].set_xlabel('Stress Level')
    axes[i].set_ylabel(col)
plt.suptitle('Psychometric Feature Distributions by Stress Level', fontsize=14)
plt.tight_layout()
plt.show()

**Finding:** Students with **High** stress exhibit significantly elevated anxiety and depression scores, lower self-esteem and sleep quality, and less social support. These patterns validate the psychometric scales as predictive features.

### 4.6 Pairplot

In [ ]:
keep = ['anxiety_level', 'depression', 'sleep_quality', 'self_esteem', 'stress_level']
sample = df[keep].sample(500, random_state=42)
g = sns.pairplot(sample, hue='stress_level', palette=palette,
                 hue_order=['Low', 'Medium', 'High'],
                 plot_kws={'alpha': 0.45, 's': 18}, diag_kind='kde')
g.fig.suptitle('Pairplot of Key Psychometric Features by Stress Level', y=1.02, fontsize=13)
plt.show()

### 4.7 Engineered Feature Distributions

In [ ]:
fig, axes = plt.subplots(1, len(engineered_cols), figsize=(18, 4))
for ax, col in zip(axes, engineered_cols):
    ax.hist(df[col].dropna(), bins=30, color='mediumpurple', edgecolor='white', alpha=0.8)
    ax.set_title(col, fontsize=9)
    ax.set_xlabel('Value')
plt.suptitle('Engineered Feature Distributions', fontsize=13)
plt.tight_layout()
plt.show()

### 4.8 Psychological Burden vs Stress Level

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for lvl in ['Low', 'Medium', 'High']:
    subset = df[df['stress_level'] == lvl]['psychological_burden'].dropna()
    ax.hist(subset, bins=30, alpha=0.6, label=lvl, color=palette[lvl], edgecolor='white')
ax.set_title('Psychological Burden Distribution by Stress Level', fontsize=13)
ax.set_xlabel('Psychological Burden Score')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.show()

**Finding:** `psychological_burden` shows clear separation between the three stress classes — it is one of the most predictive engineered features. High-stress students cluster at higher values while low-stress students concentrate near zero.

## 5. Train/Test Split, Imputation & Scaling

The global 80/20 split is performed **before** any statistics are fitted to prevent data leakage:

1. **`fit_transform_cleaning`** — fits mean imputer and IQR outlier capper on the training set, applies both to train and test.
2. **`apply_encoding_and_scaling`** — fits MinMaxScaler on the training set only.
3. **`transform_with_scaler`** — applies the already-fitted scaler to the test set.

In [ ]:
from sklearn.model_selection import train_test_split
from preprocessing import fit_transform_cleaning, apply_encoding_and_scaling, transform_with_scaler

df_train, df_test = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['stress_level_encoded']
)
print(f'Train: {len(df_train)} rows | Test: {len(df_test)} rows')

# Fit imputer + outlier capper on train only
df_train, df_test = fit_transform_cleaning(df_train, df_test, artifacts)

# Fit scaler on train only
df_train, scaler = apply_encoding_and_scaling(df_train, artifacts)
df_test = transform_with_scaler(df_test, artifacts)

print(f'Post-scaling train shape: {df_train.shape}')
df_train.head()

## 6. Clustering Preview

K is selected automatically using silhouette score (range 2–6). Both elbow (inertia) and silhouette curves are shown.

In [ ]:
from clustering import get_feature_cols, fit_kmeans

feature_cols = get_feature_cols(df_train)
print(f'{len(feature_cols)} feature columns: {feature_cols}')

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

X = df_train[feature_cols].values
k_range = range(2, 7)
inertias, silhouettes = [], []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X, labels, sample_size=min(2000, len(X)), random_state=42))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ks = list(k_range)
ax1.plot(ks, inertias, 'bo-', markersize=8, linewidth=2)
ax1.set_xlabel('K'); ax1.set_ylabel('Inertia (WCSS)'); ax1.set_title('Elbow Curve'); ax1.grid(True, alpha=0.3)

best_k = ks[int(np.argmax(silhouettes))]
ax2.plot(ks, silhouettes, 'rs-', markersize=8, linewidth=2)
ax2.axvline(best_k, color='green', linestyle='--', alpha=0.8, label=f'Best K={best_k}')
ax2.set_xlabel('K'); ax2.set_ylabel('Silhouette Score'); ax2.set_title('Silhouette Score by K')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.suptitle('K Selection: Elbow + Silhouette', fontsize=13)
plt.tight_layout()
plt.show()
print(f'Optimal K = {best_k}')

In [ ]:
from sklearn.decomposition import PCA

df_train, kmeans = fit_kmeans(df_train, feature_cols, n_clusters=best_k)
df_test = df_test.copy()
df_test['cluster'] = kmeans.predict(df_test[feature_cols])

pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(df_train[feature_cols])
v1, v2 = pca.explained_variance_ratio_

colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']
fig, ax = plt.subplots(figsize=(9, 6))
for cid in range(best_k):
    mask = df_train['cluster'].values == cid
    ax.scatter(coords[mask, 0], coords[mask, 1], c=colors[cid % len(colors)],
               label=f'Cluster {cid} (n={mask.sum()})', alpha=0.55, s=25)
ax.set_xlabel(f'PC1 ({v1:.1%} var)')
ax.set_ylabel(f'PC2 ({v2:.1%} var)')
ax.set_title('K-Means Clusters — PCA 2-D Projection (Training Set)')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Key Findings Summary

| Finding | Insight |
|---|---|
| **Anxiety & Depression** | Strongest positive predictors of high stress |
| **Self-Esteem** | Acts as a protective buffer — inversely correlated with stress |
| **Sleep Quality** | Progressively lower as stress level increases |
| **Social Support** | Below-average values consistently co-occur with burnout |
| **Psychological Burden** | Engineered composite cleanly separates the three stress classes |
| **Support Deficit** | Captures the gap between demands and available support |
| **Clustering** | Silhouette-optimal K=4 finds 2 pure clusters (Low/High) and 1 mixed cluster |
| **No Data Leakage** | Imputer, capper, and scaler all fitted on training set only |
| **Hybrid vs Baseline** | Hybrid +2.3% accuracy over global RF; McNemar p=0.30 (not significant at 95%) |

---
*Run `python main.py` from the project root to execute the full pipeline and generate all saved outputs.*